#### Imports (bibliotecas que vamos usar no projeto inteiro)

##### ============================================================
##### IMPORTAÇÃO DE BIBLIOTECAS
Aqui importamos tudo que vamos usar no projeto inteiro
Fazemos isso no começo para que todas as células abaixo
já tenham acesso a essas ferramentas
##### ============================================================

In [87]:
# Pandas: biblioteca para trabalhar com tabelas de dados (DataFrames)
# É como uma planilha do Excel dentro do Python
# Apelido "pd" para escrever menos
import pandas as pd

# NumPy: biblioteca para cálculos matemáticos com listas de números
# Muito mais rápida que fazer contas com loops normais do Python
# Apelido "np" para escrever menos
import numpy as np

# datetime: módulo nativo do Python para trabalhar com datas
# - datetime: cria objetos de data (ex: 1 de janeiro de 2024)
# - timedelta: representa uma duração (ex: 30 dias)
from datetime import datetime, timedelta

# random: módulo nativo para gerar números e escolhas aleatórias
# Usamos para simular vendas fictícias que pareçam reais
import random

# os: módulo nativo para interagir com o sistema operacional
# Usamos para criar pastas e verificar se arquivos existem
import os

# re: módulo nativo para expressões regulares
# Expressões regulares são padrões para buscar/limpar textos
import re

# json: módulo nativo para ler e escrever arquivos JSON
# JSON é um formato de dados muito usado em APIs e configurações
import json

# matplotlib.pyplot: biblioteca para criar gráficos
# Apelido "plt" para escrever menos
import matplotlib.pyplot as plt

# seaborn: biblioteca que deixa os gráficos mais bonitos
# Funciona em cima do matplotlib, adicionando estilos prontos
# Apelido "sns" para escrever menos
import seaborn as sns

# Mensagem para confirmar que tudo importou sem erro
print("Todas as bibliotecas importadas com sucesso!")



Todas as bibliotecas importadas com sucesso!


In [88]:
# Agora vamos ler o arquivo CSV e fazer a importação com os dados de vendas

df = pd.read_csv('vendas.csv')




###### ============================================================
#### RF02: Inspecionar e Descrever os Dados

Antes de limpar os dados, precisamos entender o que temos:
quantas linhas, quais colunas, quais tipos, onde estão os erros
É como abrir uma planilha pela primeira vez e dar uma olhada geral
###### ============================================================

In [89]:
def inspecionar_dados(df):
    """Exibe informações básicas do DataFrame."""

    print("\n=== INSPEÇÃO INICIAL DO DATASET ===")

    # shape mostra (número de linhas, número de colunas)
    # Esperamos (200, 8) = 200 vendas com 8 informações cada
    print(f"Shape: {df.shape}")

    # list(df.columns) mostra o nome de todas as colunas
    print(f"\nColunas: {list(df.columns)}")

    # dtypes mostra o tipo de dado de cada coluna
    # object = texto, int64 = número inteiro, float64 = número decimal
    print(f"\nTipos de dados:\n{df.dtypes}")

    # isnull().sum() conta quantos valores nulos tem em cada coluna
    # Valores nulos são os None que colocamos propositalmente no RF01
    print(f"\nValores nulos por coluna:\n{df.isnull().sum()}")

    # head() mostra as 5 primeiras linhas da tabela
    # Útil para ter uma noção visual dos dados
    print(f"\nPrimeiros registros:\n{df.head()}")

    # describe() calcula estatísticas automáticas das colunas numéricas
    # Mostra: contagem, média, desvio padrão, mínimo, máximo, quartis
    print(f"\nEstatísticas descritivas:\n{df.describe()}")

# Chama a função passando nosso dataset bruto
inspecionar_dados(df)
     




=== INSPEÇÃO INICIAL DO DATASET ===
Shape: (200, 8)

Colunas: ['id_venda', 'data_venda', 'cliente', 'produto', 'categoria', 'regiao', 'quantidade', 'preco_unitario']

Tipos de dados:
id_venda            int64
data_venda            str
cliente               str
produto               str
categoria             str
regiao                str
quantidade        float64
preco_unitario    float64
dtype: object

Valores nulos por coluna:
id_venda           0
data_venda         0
cliente            0
produto            0
categoria          0
regiao             0
quantidade         6
preco_unitario    10
dtype: int64

Primeiros registros:
   id_venda  data_venda      cliente     produto     categoria    regiao  \
0         1  2024-05-20  Cliente_035       Mouse   Periféricos   Sudeste   
1         2  2024-02-17  Cliente_042     Teclado   Periféricos     Norte   
2         3  2024-05-22  Cliente_022     Monitor  Computadores  Nordeste   
3         4  2024-06-21  Cliente_017  Smartphone     Celular

###### ============================================================
#### RF03 – LIMPAR E TRATAR OS DADOS
 Aqui tratamos todos os problemas que colocamos de propósito:
- Espaços extras nos nomes de produtos
- Datas inválidas ("DATA INVÁLIDA")
- Valores nulos em quantidade e preço
É uma das etapas mais importantes em ciência de dados
###### ============================================================

In [90]:


# Recarrega os dados brutos do CSV para garantir que está limpo
df_bruto = pd.read_csv("vendas.csv")

def limpar_dados(df):
    """
    Limpa e trata o DataFrame de vendas.
    Retorna o DataFrame limpo e um relatório de limpeza.
    """

    # Guarda a quantidade inicial de linhas para comparar depois
    n_inicial = len(df)

    # Dicionário vazio que vai armazenar o relatório de limpeza
    relatorio = {}

    # --- PASSO 1: Remover espaços extras em colunas de texto ---
    colunas_texto = df.select_dtypes(include=["object", "string"]).columns
    for col in colunas_texto:
        df[col] = df[col].str.strip()

    # --- PASSO 2: Converter datas e remover as inválidas ---
    df["data_venda"] = pd.to_datetime(df["data_venda"], errors="coerce")
    n_datas_invalidas = df["data_venda"].isnull().sum()
    df = df.dropna(subset=["data_venda"])
    relatorio["datas_invalidas_removidas"] = n_datas_invalidas

    # --- PASSO 3: Remover linhas com quantidade ou preço nulos ---
    n_antes = len(df)
    df = df.dropna(subset=["quantidade", "preco_unitario"])
    relatorio["linhas_nulas_removidas"] = n_antes - len(df)

    # --- PASSO 4: Garantir tipos numéricos corretos ---
    df["quantidade"] = df["quantidade"].astype(int)
    df["preco_unitario"] = df["preco_unitario"].astype(float)

    n_final = len(df)
    relatorio["registros_iniciais"] = n_inicial
    relatorio["registros_finais"] = n_final
    relatorio["registros_removidos_total"] = n_inicial - n_final

    print("\n=== RELATÓRIO DE LIMPEZA ===")
    for chave, valor in relatorio.items():
        print(f"  {chave}: {valor}")

    return df, relatorio

# Executa a limpeza usando uma cópia do df_bruto
df_limpo, relatorio = limpar_dados(df_bruto.copy())


=== RELATÓRIO DE LIMPEZA ===
  datas_invalidas_removidas: 4
  linhas_nulas_removidas: 16
  registros_iniciais: 200
  registros_finais: 180
  registros_removidos_total: 20


##### Validação da Limpeza

In [91]:
# Confirma que a limpeza funcionou
print("Valores nulos após limpeza:")
print(df_limpo.isnull().sum())
print(f"\nLinhas antes: {relatorio['registros_iniciais']}")
print(f"Linhas depois: {relatorio['registros_finais']}")
print(f"Removidas: {relatorio['registros_removidos_total']}")
print(f"\nTipo da coluna data_venda: {df_limpo['data_venda'].dtype}")
print(f"\nProdutos únicos: {df_limpo['produto'].unique()}")

Valores nulos após limpeza:
id_venda          0
data_venda        0
cliente           0
produto           0
categoria         0
regiao            0
quantidade        0
preco_unitario    0
dtype: int64

Linhas antes: 200
Linhas depois: 180
Removidas: 20

Tipo da coluna data_venda: datetime64[us]

Produtos únicos: <StringArray>
['Mouse', 'Teclado', 'Smartphone', 'Monitor', 'Headset', 'Tablet', 'Notebook']
Length: 7, dtype: str


##### ============================================================
##### RF04 – CRIAR COLUNAS DERIVADAS COM TRANSFORMAÇÕES
Colunas derivadas são colunas novas criadas a partir das existentes
Exemplo: receita_total = quantidade x preço
Isso enriquece os dados para análises mais detalhadas
##### ============================================================

In [92]:
def criar_colunas_derivadas(df):
    """Cria colunas calculadas e derivadas a partir do dataset limpo."""

    # Receita total = quantidade vezes preço unitário
    # O Pandas faz a multiplicação linha por linha automaticamente
    df["receita_total"] = df["quantidade"] * df["preco_unitario"]

    # Extrai o número do mês da data (1 = janeiro, 12 = dezembro)
    # .dt.month acessa o componente "mês" de uma coluna de datas
    df["mes"] = df["data_venda"].dt.month

    # Extrai o nome do mês por extenso (January, February...)
    # %B é o código de formatação para nome completo do mês
    df["mes_nome"] = df["data_venda"].dt.strftime("%B")

    # Cria a coluna de trimestre: Q1, Q2, Q3 ou Q4
    # .dt.quarter retorna 1, 2, 3 ou 4
    # .apply(lambda q: f"Q{q}") transforma 1 em "Q1", 2 em "Q2" etc.
    # lambda é uma função anônima (sem nome) usada em uma linha só
    df["trimestre"] = df["data_venda"].dt.quarter.apply(lambda q: f"Q{q}")

    # Extrai o ano da data (2024)
    df["ano"] = df["data_venda"].dt.year

    # Classificação da receita por item usando np.select
    # np.select é uma forma vetorizada de fazer if/elif/else
    # "Vetorizada" significa que aplica em todas as linhas de uma vez
    # sem precisar de um loop for
    condicoes = [
        df["receita_total"] < 500,
        (df["receita_total"] >= 500) & (df["receita_total"] < 5000),
        df["receita_total"] >= 5000
    ]
    classificacoes = ["Baixo Valor", "Médio Valor", "Alto Valor"]

    # np.select aplica: se condição 1 usa classificação 1 etc.
    # default é o valor caso nenhuma condição seja verdadeira
    df["faixa_receita_item"] = np.select(condicoes, classificacoes,
                                         default="Não Classificado")

    print("\n=== COLUNAS DERIVADAS CRIADAS ===")
    print(df[["data_venda", "receita_total", "mes", "trimestre",
              "faixa_receita_item"]].head())

    return df

# Aplica as transformações no DataFrame limpo
df_limpo = criar_colunas_derivadas(df_limpo)


=== COLUNAS DERIVADAS CRIADAS ===
  data_venda  receita_total  mes trimestre faixa_receita_item
0 2024-05-20         205.80    5        Q2        Baixo Valor
1 2024-02-17        1504.16    2        Q1        Médio Valor
3 2024-06-21       10007.04    6        Q2         Alto Valor
4 2024-07-12         970.40    7        Q3        Médio Valor
5 2024-04-26        3800.48    4        Q2        Médio Valor


###### ============================================================
#### RF05 – CALCULAR MÉTRICAS AGREGADAS (groupby)
 groupby agrupa os dados por uma coluna e calcula estatísticas
 É como criar uma tabela dinâmica no Excel
 Exemplo: "qual a receita total de cada mês?"
###### ============================================================

In [93]:
def calcular_metricas(df):
    """Calcula e retorna métricas agregadas do dataset."""

    # Dicionário vazio para guardar todas as tabelas de métricas
    metricas = {}

    # --- Receita por mês ---
    # groupby("mes") agrupa todas as vendas pelo número do mês
    # .agg() permite calcular múltiplas estatísticas de uma vez
    # ("receita_total", "sum") = soma todas as receitas daquele mês
    # ("quantidade", "sum") = soma todas as quantidades daquele mês
    # ("id_venda", "count") = conta quantas vendas teve naquele mês
    por_mes = df.groupby("mes").agg(
        receita_total=("receita_total", "sum"),
        quantidade=("quantidade", "sum"),
        n_vendas=("id_venda", "count")
    ).reset_index().sort_values("mes")
    # reset_index() transforma o agrupamento em tabela normal
    # sort_values("mes") ordena por mês (1, 2, 3...)
    metricas["por_mes"] = por_mes

    # --- Top 5 produtos por receita ---
    # Agrupa por produto, soma a receita de cada um
    # ordena do maior para o menor e pega os 5 primeiros
    top_produtos = df.groupby("produto")["receita_total"].sum()\
                     .sort_values(ascending=False).head(5).reset_index()
    metricas["top_produtos"] = top_produtos

    # --- Receita por categoria ---
    # Mesmo conceito: agrupa por categoria e soma a receita
    por_categoria = \
        df.groupby("categoria")["receita_total"].sum().reset_index()
    metricas["por_categoria"] = por_categoria

    # --- Receita por região ---
    # Calcula receita total e ticket médio (média por venda) por região
    por_regiao = df.groupby("regiao").agg(
        receita_total=("receita_total", "sum"),
        media_ticket=("receita_total", "mean")
    ).reset_index().sort_values("receita_total", ascending=False)
    metricas["por_regiao"] = por_regiao

    # Exibe todas as tabelas de métricas
    for nome, tabela in metricas.items():
        print(f"\n=== {nome.upper().replace('_', ' ')} ===")
        print(tabela.to_string(index=False))

    return metricas

# Calcula todas as métricas
metricas = calcular_metricas(df_limpo)


=== POR MES ===
 mes  receita_total  quantidade  n_vendas
   1       47169.34          45         9
   2       69968.85          91        15
   3      124532.20          88        17
   4      103611.52          75        15
   5      106602.03          89        14
   6      117334.85          75        15
   7      110935.66          99        19
   8      140320.55          81        13
   9       90627.26          63        10
  10      137350.13         118        20
  11       69554.97          65        12
  12      115925.56         105        21

=== TOP PRODUTOS ===
   produto  receita_total
    Tablet      353176.81
  Notebook      350212.88
Smartphone      234977.18
   Monitor      190466.41
   Headset       58661.68

=== POR CATEGORIA ===
   categoria  receita_total
   Celulares      588153.99
Computadores      540679.29
 Periféricos      105099.64

=== POR REGIAO ===
      regiao  receita_total  media_ticket
     Sudeste      329458.34   8447.649744
Centro-Oeste      24